# 6.4 · DBSCAN / Density-Based Spatial Clustering

> **课程定位 / Where this fits**
> K-Means/层次都假设簇是团状/球形, 还得(或间接)定簇数, 也不会处理噪声。**DBSCAN** 换了范式: 簇 = **高密度区域**, 由密度连通定义。它能找**任意形状**的簇、**自动识别噪声/离群点**、**不需预设簇数**。月牙、环形这类 K-Means 的死穴, DBSCAN 轻松搞定。
> DBSCAN defines clusters as dense regions: arbitrary shapes, automatic noise detection, no preset cluster count.

> 💡 **面试相关 / Interview-relevant**
> - "DBSCAN 的核心/边界/噪声点定义" ★★★★★
> - "eps 和 min_samples 怎么调(k-distance 图)" ★★★★★
> - "DBSCAN 优缺点 vs K-Means" ★★★★★
> - "DBSCAN 为什么对不同密度的簇会失败" ★★★★（引出 HDBSCAN）
> - "DBSCAN 复杂度 / 高维表现" ★★★

---

## 学习目标 / Learning Objectives
1. 核心点/边界点/噪声点 + 密度可达。
2. 从零实现 DBSCAN(看清密度连通)。
3. **eps / min_samples** 调参(k-distance 肘部)。
4. 任意形状 + 噪声识别的威力。
5. 不同密度簇的局限(为 6.5 HDBSCAN 铺垫)。

## 目录 / TOC
1. [核心/边界/噪声 + 密度可达 ⭐](#1)
2. [🌙 数据: moons + 从零实现 ⭐](#2)
3. [eps/min_samples 调参 ⭐](#3)
4. [任意形状 + 噪声 vs K-Means](#4)
5. [不同密度的局限 ⭐](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 核心/边界/噪声 + 密度可达 ⭐ / Core, Border, Noise

两个超参: **eps(ε, 邻域半径)** 和 **min_samples(成核所需邻居数)**。据此把每个点分三类:
- **核心点(core)**: ε 邻域内至少有 min_samples 个点(含自己)→ 处在稠密区。
- **边界点(border)**: 自己不够核心, 但落在某核心点的 ε 邻域内。
- **噪声点(noise)**: 既非核心也非边界 → 离群, 标签 −1。

**簇的形成**: 从一个核心点出发, 把所有**密度可达**(经由一串核心点的 ε 邻域链相连)的点纳入同一簇。簇因此能沿密度"长"成任意形状。无需指定簇数——簇数由数据密度自然涌现。


<a id="2"></a>
## 2. 数据: moons + 从零实现 ⭐ / Data & From Scratch

**make_moons**: 两个交错的半月形, 线性不可分、非球形——DBSCAN 的主场, K-Means 的噩梦。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons
sns.set_theme(style="whitegrid")

X, _ = make_moons(300, noise=0.06, random_state=0)
# 加几个离群噪声点 / add outliers
rng = np.random.default_rng(0)
X = np.vstack([X, rng.uniform(-1.5, 2.5, (12, 2))])
print(f"moons + 噪声: {X.shape}")

def dbscan_scratch(X, eps, min_samples):
    n = len(X); labels = np.full(n, -1); cluster = 0
    D = np.sqrt(((X[:,None,:]-X[None,:,:])**2).sum(-1))   # 距离矩阵
    visited = np.zeros(n, bool)
    for i in range(n):
        if visited[i]: continue
        visited[i] = True
        nbrs = np.where(D[i] <= eps)[0]
        if len(nbrs) < min_samples:
            continue                      # 暂标噪声(可能后续变边界)
        labels[i] = cluster               # 开新簇
        seeds = list(nbrs)
        k = 0
        while k < len(seeds):
            j = seeds[k]; k += 1
            if labels[j] == -1: labels[j] = cluster   # 噪声→边界, 收编
            if not visited[j]:
                visited[j] = True
                jn = np.where(D[j] <= eps)[0]
                if len(jn) >= min_samples:             # j 也是核心→扩张
                    seeds += list(jn)
        cluster += 1
    return labels

lab_scratch = dbscan_scratch(X, eps=0.2, min_samples=5)
from sklearn.cluster import DBSCAN
lab_sk = DBSCAN(eps=0.2, min_samples=5).fit_predict(X)
print(f"从零 DBSCAN: {len(set(lab_scratch))-(1 if -1 in lab_scratch else 0)} 簇, {(lab_scratch==-1).sum()} 噪声")
print(f"sklearn   : {len(set(lab_sk))-(1 if -1 in lab_sk else 0)} 簇, {(lab_sk==-1).sum()} 噪声")
from sklearn.metrics import adjusted_rand_score
print(f"两者一致性 ARI = {adjusted_rand_score(lab_scratch, lab_sk):.3f}")


<a id="3"></a>
## 3. eps/min_samples 调参 ⭐ / Tuning

DBSCAN 对 **eps** 敏感。经验法子: 画 **k-distance 图**——每个点到第 k(=min_samples)近邻的距离, 升序排列。曲线的**肘部**(陡升处)就是好的 eps: 肘部以下是簇内距离, 以上是跨簇/噪声。


In [ ]:
from sklearn.neighbors import NearestNeighbors
k = 5
nn = NearestNeighbors(n_neighbors=k).fit(X)
kdist = np.sort(nn.kneighbors(X)[0][:, -1])    # 到第k近邻距离, 升序
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(kdist)
ax.axhline(0.2, color="r", ls="--", label="肘部 eps≈0.2")
ax.set_xlabel("点(按 k-distance 升序)"); ax.set_ylabel(f"到第{k}近邻距离"); ax.legend()
ax.set_title("k-distance 图: 肘部(陡升处)即合适 eps")
plt.tight_layout(); plt.show()
print("eps 取肘部值; min_samples 经验 ≈ 2×维度, 噪声多则调大")


<a id="4"></a>
## 4. 任意形状 + 噪声 vs K-Means / Arbitrary Shapes & Noise


In [ ]:
from sklearn.cluster import KMeans
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
mask = lab_sk == -1
axes[0].scatter(X[~mask,0], X[~mask,1], c=lab_sk[~mask], cmap="coolwarm", s=18)
axes[0].scatter(X[mask,0], X[mask,1], c="k", marker="x", s=40, label="噪声(-1)")
axes[0].set_title("DBSCAN: 两条月牙 + 自动标出噪声"); axes[0].legend()

km = KMeans(2, n_init=10, random_state=0).fit_predict(X)
axes[1].scatter(X[:,0], X[:,1], c=km, cmap="coolwarm", s=18)
axes[1].set_title("K-Means: 强行球形切分, 切错且无噪声概念")
plt.tight_layout(); plt.show()
print("DBSCAN 抓住非球形簇并隔离离群点; K-Means 只能球形划分、把噪声硬塞进簇")


<a id="5"></a>
## 5. 不同密度的局限 ⭐ / The Varying-Density Limitation

DBSCAN 用**单一全局 eps**。若数据里有的簇稠密、有的稀疏, **一个 eps 无法同时合适**: eps 小→稀疏簇被打散成噪声; eps 大→稠密的小簇被合并。这是 DBSCAN 的根本短板, 也正是 **HDBSCAN(6.5)** 要解决的——用层次化的密度, 适应不同密度。


In [ ]:
from sklearn.cluster import DBSCAN
# 造两个密度差异大的簇 / one dense + one sparse cluster
rng = np.random.default_rng(1)
dense = rng.normal([0,0], 0.3, (200,2))
sparse = rng.normal([3,3], 1.0, (200,2))
Xd = np.vstack([dense, sparse])
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, eps in zip(axes, [0.3, 0.6, 1.0]):
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xd)
    nnoise = (lab==-1).sum(); nclu = len(set(lab))-(1 if -1 in lab else 0)
    m = lab==-1
    ax.scatter(Xd[~m,0], Xd[~m,1], c=lab[~m], cmap="tab10", s=12)
    ax.scatter(Xd[m,0], Xd[m,1], c="k", marker="x", s=20)
    ax.set_title(f"eps={eps}: {nclu}簇, {nnoise}噪声")
plt.suptitle("单一全局 eps 难兼顾不同密度: 小eps 拆散稀疏簇, 大eps 合并稠密簇")
plt.tight_layout(); plt.show()
print("没有一个 eps 能同时正确处理稠密簇和稀疏簇 → HDBSCAN(6.5)用层次密度解决")


<a id="6"></a>
## 6. 小结 / Summary

```
DBSCAN: 簇=密度连通的高密区域; 两超参 eps(邻域半径) + min_samples(成核阈值)
点分三类: 核心(邻域≥min_samples) / 边界(在核心邻域内) / 噪声(-1)
密度可达: 从核心点经核心链扩张 → 任意形状簇, 自动识别噪声, 不预设簇数
调 eps: k-distance 图的肘部; min_samples ≈ 2×维度
局限: 单一全局 eps 无法兼顾不同密度的簇 → HDBSCAN(6.5)
```

### 💡 面试速查
1. **核心/边界/噪声**三类点; 簇由密度可达连通成任意形状
2. **eps**(k-distance 肘部) + **min_samples**(≈2×维度)
3. **优点**: 任意形状、自动噪声、不预设K; **缺点**: 对eps敏感、怕不同密度、高维退化
4. **不同密度失败** → HDBSCAN
5. **vs K-Means**: DBSCAN 非球形+噪声, K-Means 球形+快+可扩展

### 下一节
**6.5 HDBSCAN**——把 DBSCAN 层次化: 在所有密度尺度上构建层次, 自动选出最稳定的簇, 不再需要手调单一 eps。
